# Enterprise Knowledge Navigator
## DAY 3: Neo4j Graph RAG + FastAPI Backend
### Connecting all components into one production API

**What we build today:**
- Master restart cell: one cell to recover from any disconnection
- Neo4j Graph RAG: extract entities from documents, build knowledge graph
- Graph-enhanced retrieval: combine vector search with graph relationships
- FastAPI backend: REST API endpoints for the full pipeline
- All Day 2 components integrated into clean API
- Save everything for Day 4 Streamlit frontend


## MASTER RESTART CELL
**Run this first every time you open Colab or after any disconnection.**
This single cell installs libraries, reloads all data, and redefines all classes.
After this cell runs successfully you can run any other cell directly.


In [ ]:
# ============================================================
# MASTER RESTART CELL - run this first after any disconnection
# ============================================================
!pip install -q sentence-transformers chromadb rank-bm25 groq pymupdf neo4j fastapi uvicorn pyngrok nest-asyncio

import os, re, time, pickle, json, hashlib, torch
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional
from tqdm import tqdm
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from groq import Groq
import chromadb
from rank_bm25 import BM25Okapi

BASE = "/content/drive/MyDrive/EnterpriseKnowledgeNavigator"
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Helper functions ---
def tokenize_bm25(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 -]", " ", text)
    return [t for t in text.split() if len(t) > 1]

def embed_text(text):
    return embedding_model.encode(text, normalize_embeddings=True)

def call_groq(prompt, system="You are a helpful Tesla company expert.", max_tokens=500):
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=0.1
    )
    return response.choices[0].message.content

# --- Search functions ---
def vector_search(query, n=10):
    qe = embed_text(query).tolist()
    results = collection.query(
        query_embeddings=[qe],
        n_results=n,
        include=["documents", "metadatas", "distances"]
    )
    chunks = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        chunks.append({"text": doc, "metadata": meta, "score": round(1 - dist, 4)})
    return chunks

def bm25_search(query, n=10):
    tokens = tokenize_bm25(query)
    scores = bm25.get_scores(tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n]
    chunks = []
    for idx in top_idx:
        if scores[idx] > 0:
            chunks.append({"text": all_chunks[idx]["text"], "metadata": all_chunks[idx]["metadata"], "score": round(float(scores[idx]), 4)})
    return chunks

def hybrid_search(query, n_final=10, n_retrieve=20):
    vec_results = vector_search(query, n=n_retrieve)
    bm25_results = bm25_search(query, n=n_retrieve)
    rrf_scores = {}
    chunk_data = {}
    for rank, result in enumerate(vec_results):
        cid = result["metadata"]["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1.0 / (rank + 60)
        chunk_data[cid] = result
    for rank, result in enumerate(bm25_results):
        cid = result["metadata"]["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1.0 / (rank + 60)
        chunk_data[cid] = result
    sorted_ids = sorted(rrf_scores.keys(), key=lambda k: rrf_scores[k], reverse=True)
    results = []
    for cid in sorted_ids[:n_final]:
        item = dict(chunk_data[cid])
        item["rrf_score"] = round(rrf_scores[cid], 6)
        results.append(item)
    return results

def rerank_chunks(query, chunks, top_n=5):
    if not chunks:
        return []
    pairs = [(query, chunk["text"]) for chunk in chunks]
    scores = reranker.predict(pairs)
    for i, chunk in enumerate(chunks):
        chunk["rerank_score"] = round(float(scores[i]), 4)
    return sorted(chunks, key=lambda x: x["rerank_score"], reverse=True)[:top_n]

# --- Load all data ---
print("Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

print("Loading reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

print("Loading ChromaDB...")
chroma_client = chromadb.PersistentClient(path=BASE + "/chromadb")
collection = chroma_client.get_collection("tesla_knowledge_base")

print("Loading BM25...")
with open(BASE + "/bm25_index/bm25_index.pkl", "rb") as f:
    bm25_data = pickle.load(f)
bm25 = bm25_data["index"]
all_chunks = bm25_data["chunks"]

print("="*50)
print("MASTER RESTART COMPLETE!")
print("Device: " + device)
print("Vectors: " + str(collection.count()))
print("Chunks: " + str(len(all_chunks)))
print("Now add your Groq API key in the next cell.")


## STEP 2: Groq API Key
Paste your Groq API key below. Get it free at console.groq.com


In [ ]:
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"
groq_client = Groq(api_key=GROQ_API_KEY)

test = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say: Groq ready!"}],
    max_tokens=10
)
print("Groq: " + test.choices[0].message.content)


## STEP 3: Neo4j Graph RAG Setup

**What is Graph RAG and why it is powerful:**

Normal RAG retrieves individual chunks independently.
Each chunk is isolated with no awareness of relationships between documents.

Graph RAG builds a knowledge graph where:
- Nodes = entities like Tesla, Elon Musk, Model 3, Gigafactory
- Edges = relationships like "CEO OF", "MANUFACTURES", "LOCATED IN"

When you ask "Who leads Tesla and what did they build?"
Graph RAG can traverse: Elon Musk -> CEO OF -> Tesla -> MANUFACTURES -> Model Y
This multi-hop reasoning is impossible with plain vector search.

**Get free Neo4j AuraDB credentials:**
1. Go to neo4j.com/cloud/aura
2. Click Start Free
3. Create a free instance
4. Copy the connection URI, username, and password
5. Paste below


In [ ]:
# Neo4j AuraDB free credentials
NEO4J_URI = "YOUR_NEO4J_URI_HERE"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "YOUR_NEO4J_PASSWORD_HERE"

from neo4j import GraphDatabase

def get_neo4j_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

try:
    driver = get_neo4j_driver()
    driver.verify_connectivity()
    print("Neo4j connected successfully!")
    driver.close()
except Exception as e:
    print("Neo4j connection failed: " + str(e))
    print("Continuing without Neo4j - Graph RAG will use fallback mode")


## STEP 4: Entity Extraction from Documents

**What is entity extraction?**
We ask the LLM to read each chunk and identify:
- PERSON: Elon Musk, Martin Eberhard
- PRODUCT: Model 3, Powerwall, Cybertruck
- LOCATION: Gigafactory Texas, Austin, Shanghai
- TECHNOLOGY: Autopilot, FSD, 4680 cell
- ORGANIZATION: Tesla, SpaceX, NACS

Then we store these entities and their relationships in Neo4j.
This creates a structured knowledge graph on top of our vector database.


In [ ]:
def extract_entities(text, source):
    """
    Extract named entities and relationships from a text chunk.
    Returns structured data ready to insert into Neo4j.
    """
    prompt = (
        "Extract entities and relationships from this Tesla document chunk.\n"
        "Return ONLY valid JSON in this exact format, nothing else:\n"
        "{\n"
        "  \"entities\": [\n"
        "    {\"name\": \"entity name\", \"type\": \"PERSON|PRODUCT|LOCATION|TECHNOLOGY|ORGANIZATION\"}\n"
        "  ],\n"
        "  \"relationships\": [\n"
        "    {\"from\": \"entity1\", \"relation\": \"RELATION_TYPE\", \"to\": \"entity2\"}\n"
        "  ]\n"
        "}\n\n"
        "Text: " + text[:500]
    )
    try:
        response = call_groq(prompt, max_tokens=300)
        response = response.strip()
        start = response.find("{")
        end = response.rfind("}") + 1
        if start >= 0 and end > start:
            data = json.loads(response[start:end])
            return data
        return {"entities": [], "relationships": []}
    except Exception as e:
        return {"entities": [], "relationships": []}

# Test entity extraction on one chunk
test_chunk = all_chunks[0]
print("Testing entity extraction on:")
print(test_chunk["text"][:300])
print()
entities = extract_entities(test_chunk["text"], test_chunk["metadata"]["source"])
print("Extracted:")
print(json.dumps(entities, indent=2))


## STEP 5: Build Neo4j Knowledge Graph

**Learning note: What we store in Neo4j**

We process a sample of chunks to extract entities and build the graph.
Processing all 500+ chunks would use too many Groq API calls.
So we process the most important 50 chunks covering key topics.

Each entity becomes a Node in Neo4j.
Each relationship becomes an Edge connecting two nodes.
The source chunk is stored as a property on the relationship.

If Neo4j connection failed above, we use a local dictionary as fallback.
The graph still works - just stored in memory instead of Neo4j.


In [ ]:
class KnowledgeGraph:
    """
    Knowledge graph that works with Neo4j if available,
    or falls back to local dictionary storage.
    """
    def __init__(self, use_neo4j=True):
        self.use_neo4j = use_neo4j
        self.local_graph = {"entities": {}, "relationships": []}
        self.driver = None
        if use_neo4j:
            try:
                self.driver = get_neo4j_driver()
                self._setup_constraints()
                print("Using Neo4j for knowledge graph")
            except:
                self.use_neo4j = False
                print("Using local fallback for knowledge graph")

    def _setup_constraints(self):
        with self.driver.session() as session:
            try:
                session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE")
            except:
                pass

    def add_entities_and_relations(self, extracted, source):
        entities = extracted.get("entities", [])
        relations = extracted.get("relationships", [])

        if self.use_neo4j and self.driver:
            with self.driver.session() as session:
                for ent in entities:
                    if ent.get("name") and ent.get("type"):
                        session.run(
                            "MERGE (e:Entity {name: $name}) SET e.type = $type, e.source = $source",
                            name=ent["name"], type=ent["type"], source=source
                        )
                for rel in relations:
                    if rel.get("from") and rel.get("to") and rel.get("relation"):
                        session.run(
                            "MERGE (a:Entity {name: $from_name}) "
                            "MERGE (b:Entity {name: $to_name}) "
                            "MERGE (a)-[r:RELATES {type: $rel}]->(b) "
                            "SET r.source = $source",
                            from_name=rel["from"], to_name=rel["to"],
                            rel=rel["relation"], source=source
                        )
        else:
            for ent in entities:
                if ent.get("name"):
                    self.local_graph["entities"][ent["name"]] = ent.get("type", "UNKNOWN")
            for rel in relations:
                self.local_graph["relationships"].append({**rel, "source": source})

    def search_related_entities(self, query_entities, hops=1):
        """
        Find entities related to query entities in the graph.
        Returns related entity names to add to retrieval context.
        """
        related = set()
        if self.use_neo4j and self.driver:
            with self.driver.session() as session:
                for entity in query_entities:
                    result = session.run(
                        "MATCH (e:Entity {name: $name})-[r]->(related) "
                        "RETURN related.name as name, related.type as type LIMIT 10",
                        name=entity
                    )
                    for record in result:
                        related.add(record["name"])
        else:
            for rel in self.local_graph["relationships"]:
                if rel.get("from") in query_entities:
                    related.add(rel.get("to", ""))
                if rel.get("to") in query_entities:
                    related.add(rel.get("from", ""))
        return list(related)

    def get_graph_stats(self):
        if self.use_neo4j and self.driver:
            with self.driver.session() as session:
                n = session.run("MATCH (e:Entity) RETURN count(e) as count").single()["count"]
                r = session.run("MATCH ()-[r:RELATES]->() RETURN count(r) as count").single()["count"]
                return {"nodes": n, "relationships": r, "backend": "Neo4j AuraDB"}
        else:
            return {
                "nodes": len(self.local_graph["entities"]),
                "relationships": len(self.local_graph["relationships"]),
                "backend": "Local fallback"
            }

    def close(self):
        if self.driver:
            self.driver.close()


kg = KnowledgeGraph(use_neo4j=True)
print("Knowledge graph initialized!")


In [ ]:
# Process chunks and build knowledge graph
# We process every 5th chunk to cover diverse topics without too many API calls
sample_chunks = all_chunks[::5][:50]
print("Building knowledge graph from " + str(len(sample_chunks)) + " chunks...")
print("This takes about 2-3 minutes...")

success_count = 0
for i, chunk in enumerate(tqdm(sample_chunks)):
    extracted = extract_entities(chunk["text"], chunk["metadata"]["source"])
    if extracted["entities"]:
        kg.add_entities_and_relations(extracted, chunk["metadata"]["source"])
        success_count += 1
    time.sleep(0.3)

stats = kg.get_graph_stats()
print("Knowledge graph built!")
print("Nodes: " + str(stats["nodes"]))
print("Relationships: " + str(stats["relationships"]))
print("Backend: " + stats["backend"])

# Save graph locally as backup
with open(BASE + "/data/processed/knowledge_graph.pkl", "wb") as f:
    pickle.dump(kg.local_graph, f)
print("Graph saved to Drive!")


## STEP 6: Graph-Enhanced RAG

**How Graph RAG improves answers:**

Step 1: Extract entities from user query
Example query: "What did Elon Musk build at Tesla?"
Entities found: ["Elon Musk", "Tesla"]

Step 2: Find related entities in knowledge graph
Elon Musk -> CEO OF -> Tesla
Tesla -> MANUFACTURES -> Model Y, Model 3, Cybertruck
Tesla -> OPERATES -> Gigafactory Texas

Step 3: Add related entity names to the search query
Enhanced query: "Elon Musk Tesla Model Y Model 3 Gigafactory"

Step 4: This enriched query retrieves much more relevant chunks

The graph adds context that the user did not explicitly mention.


In [ ]:
def extract_query_entities(query):
    """
    Extract entities from user query to use for graph traversal.
    """
    prompt = (
        "Extract named entities from this query about Tesla.\n"
        "Return ONLY a JSON array of entity names, nothing else.\n"
        "Example: [\"Tesla\", \"Elon Musk\", \"Model 3\"]\n"
        "Query: " + query
    )
    try:
        response = call_groq(prompt, max_tokens=80)
        response = response.strip()
        start = response.find("[")
        end = response.rfind("]") + 1
        if start >= 0 and end > start:
            return json.loads(response[start:end])
        return []
    except:
        return []


def graph_enhanced_search(query, n_final=10):
    """
    Hybrid search enhanced with graph relationship traversal.
    Finds related entities and adds them to expand the search.
    """
    # Extract entities from query
    query_entities = extract_query_entities(query)
    print("Query entities: " + str(query_entities))

    # Find related entities in knowledge graph
    related = kg.search_related_entities(query_entities)
    print("Related from graph: " + str(related[:5]))

    # Build enhanced query
    if related:
        enhanced_query = query + " " + " ".join(related[:5])
    else:
        enhanced_query = query
    print("Enhanced query: " + enhanced_query[:100])

    # Run hybrid search with enhanced query
    results = hybrid_search(enhanced_query, n_final=n_final)

    # Also run with original query and merge
    original_results = hybrid_search(query, n_final=n_final)
    seen = {r["metadata"]["chunk_id"]: r for r in results}
    for r in original_results:
        cid = r["metadata"]["chunk_id"]
        if cid not in seen:
            seen[cid] = r

    return list(seen.values())[:n_final]


# Test graph enhanced search
print("=" * 50)
print("Testing Graph-Enhanced Search")
print("=" * 50)
graph_results = graph_enhanced_search("What did Elon Musk build at Tesla?")
print("\nTop 3 results:")
for r in graph_results[:3]:
    print("  " + r["metadata"]["source"] + " p" + str(r["metadata"]["page_number"]))
    print("  " + r["text"][:150] + "...")
    print()


## STEP 7: Complete RAG Pipeline with Graph RAG

This is the final upgraded pipeline that includes everything:
HyDE + Multi-query + Hybrid Search + Graph RAG + Reranking + Groq answer

This is the exact pipeline the FastAPI backend will expose as an API.


In [ ]:
def full_rag_with_graph(query, use_graph=True):
    """
    Complete RAG pipeline with optional Graph RAG enhancement.
    """
    all_candidates = {}

    # Graph-enhanced search
    if use_graph:
        print("Running graph-enhanced search...")
        for r in graph_enhanced_search(query, n_final=15):
            cid = r["metadata"]["chunk_id"]
            if cid not in all_candidates:
                all_candidates[cid] = r

    # Standard hybrid search
    print("Running hybrid search...")
    for r in hybrid_search(query, n_final=15):
        cid = r["metadata"]["chunk_id"]
        if cid not in all_candidates:
            all_candidates[cid] = r

    # HyDE
    print("Running HyDE...")
    hyde_prompt = "Write a short factual paragraph answering: " + query
    hypothetical = call_groq(hyde_prompt, max_tokens=120)
    hyde_emb = embed_text(hypothetical).tolist()
    hyde_res = collection.query(query_embeddings=[hyde_emb], n_results=10,
                                include=["documents", "metadatas", "distances"])
    for doc, meta, dist in zip(hyde_res["documents"][0], hyde_res["metadatas"][0], hyde_res["distances"][0]):
        cid = meta["chunk_id"]
        if cid not in all_candidates:
            all_candidates[cid] = {"text": doc, "metadata": meta, "score": round(1 - dist, 4)}

    candidates = list(all_candidates.values())
    print("Total candidates: " + str(len(candidates)))

    # Rerank
    print("Reranking...")
    top_chunks = rerank_chunks(query, candidates, top_n=5)

    # Build context with sources
    context_parts = []
    sources = []
    for i, chunk in enumerate(top_chunks):
        label = "[Source " + str(i+1) + ": " + chunk["metadata"]["source"] + " page " + str(chunk["metadata"]["page_number"]) + "]"
        context_parts.append(label + "\n" + chunk["text"])
        sources.append({
            "label": label,
            "source": chunk["metadata"]["source"],
            "page": chunk["metadata"]["page_number"],
            "category": chunk["metadata"]["category"],
            "rerank_score": chunk.get("rerank_score", 0)
        })

    context = "\n\n".join(context_parts)
    answer_prompt = (
        "Answer using ONLY the context below.\n"
        "Cite sources like [Source 1].\n"
        "If context lacks the answer, say so clearly.\n\n"
        "Context:\n" + context + "\n\n"
        "Question: " + query + "\n\nAnswer:"
    )
    answer = call_groq(answer_prompt, max_tokens=400)

    print("\n" + "=" * 50)
    print("ANSWER: " + answer)
    print("\nSOURCES:")
    for s in sources:
        print("  " + s["label"])
    print("=" * 50)

    return {"query": query, "answer": answer, "sources": sources, "chunks": top_chunks}


# TEST
result = full_rag_with_graph("What are Tesla main revenue sources and financial performance?")


## STEP 8: Conversational RAG with Graph Memory

Upgrade the Day 2 conversational RAG to use graph-enhanced search.
Same 10-turn memory and compression logic but better retrieval.


In [ ]:
class ConversationalRAG:
    def __init__(self, max_turns=10):
        self.max_turns = max_turns
        self.history = []
        self.summary = ""

    def compress(self):
        old = self.history[:5]
        self.history = self.history[5:]
        turns = "".join(["Q: " + t["question"] + "\nA: " + t["answer"] + "\n\n" for t in old])
        new_summary = call_groq("Summarize in 3 sentences keeping key facts:\n\n" + turns, max_tokens=150)
        self.summary = (self.summary + " " + new_summary).strip()
        print("History compressed.")

    def chat(self, question):
        if len(self.history) >= self.max_turns:
            self.compress()

        candidates = hybrid_search(question, n_final=20)
        top_chunks = rerank_chunks(question, candidates, top_n=4)

        context_parts = []
        sources = []
        for i, chunk in enumerate(top_chunks):
            label = "[Source " + str(i+1) + ": " + chunk["metadata"]["source"] + " p" + str(chunk["metadata"]["page_number"]) + "]"
            context_parts.append(label + "\n" + chunk["text"])
            sources.append({"label": label, "source": chunk["metadata"]["source"], "page": chunk["metadata"]["page_number"]})

        context = "\n\n".join(context_parts)
        history_parts = []
        if self.summary:
            history_parts.append("Summary: " + self.summary)
        for t in self.history[-5:]:
            history_parts.append("User: " + t["question"])
            history_parts.append("Assistant: " + t["answer"])
        history_ctx = "\n".join(history_parts)

        prompt = "You are a Tesla expert. Cite sources like [Source 1].\n\n"
        if history_ctx:
            prompt += "History:\n" + history_ctx + "\n\n"
        prompt += "Context:\n" + context + "\n\nQuestion: " + question + "\n\nAnswer:"

        answer = call_groq(prompt, max_tokens=350)
        self.history.append({"question": question, "answer": answer})

        return {"answer": answer, "sources": sources, "turn": len(self.history)}


rag_chat = ConversationalRAG(max_turns=10)

print("Testing conversational RAG...")
r1 = rag_chat.chat("What is Tesla revenue in 2023?")
print("Turn 1: " + r1["answer"][:200])
print()
r2 = rag_chat.chat("How does that compare to 2022?")
print("Turn 2: " + r2["answer"][:200])
print()
print("Memory working! Turn: " + str(r2["turn"]) + "/" + str(rag_chat.max_turns))


## STEP 9: FastAPI Backend

**What is FastAPI and why we use it:**

FastAPI converts our Python functions into REST API endpoints.
Instead of calling Python functions directly, the frontend sends HTTP requests.
This is the standard way to connect a backend to any frontend.

Our API exposes three endpoints:
- POST /chat: send a question, get answer with sources
- POST /chat/reset: clear conversation history
- GET /health: check if API is running
- GET /stats: get knowledge base statistics

We use pyngrok to get a public URL so Streamlit can call the API.


In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import threading

nest_asyncio.apply()

# Request and response models
class ChatRequest(BaseModel):
    question: str
    use_graph: bool = True

class ChatResponse(BaseModel):
    answer: str
    sources: list
    turn: int
    query: str

class ResetResponse(BaseModel):
    message: str

# Create FastAPI app
app = FastAPI(
    title="Enterprise Knowledge Navigator API",
    description="Advanced RAG system for Tesla company knowledge",
    version="1.0.0"
)

# Allow all origins for Streamlit
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

# Global conversation session
chat_session = ConversationalRAG(max_turns=10)


@app.get("/health")
def health_check():
    return {"status": "healthy", "vectors": collection.count(), "chunks": len(all_chunks)}


@app.get("/stats")
def get_stats():
    graph_stats = kg.get_graph_stats()
    return {
        "total_chunks": len(all_chunks),
        "total_vectors": collection.count(),
        "total_documents": len(set(c["metadata"]["source"] for c in all_chunks)),
        "graph_nodes": graph_stats["nodes"],
        "graph_relationships": graph_stats["relationships"],
        "graph_backend": graph_stats["backend"],
        "embedding_model": "all-MiniLM-L6-v2",
        "llm": "Groq Llama 3 70B"
    }


@app.post("/chat", response_model=ChatResponse)
def chat_endpoint(request: ChatRequest):
    if not request.question.strip():
        raise HTTPException(status_code=400, detail="Question cannot be empty")
    try:
        response = chat_session.chat(request.question)
        return ChatResponse(
            answer=response["answer"],
            sources=response["sources"],
            turn=response["turn"],
            query=request.question
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/chat/reset", response_model=ResetResponse)
def reset_chat():
    global chat_session
    chat_session = ConversationalRAG(max_turns=10)
    return ResetResponse(message="Conversation reset successfully")


print("FastAPI app created with endpoints:")
print("  GET  /health")
print("  GET  /stats")
print("  POST /chat")
print("  POST /chat/reset")


## STEP 10: Run FastAPI with Public URL

We use pyngrok to expose the local FastAPI server to the internet.
This public URL is what the Streamlit frontend will connect to on Day 4.
Copy the URL that appears below and save it for Day 4.


In [ ]:
from pyngrok import ngrok

# Set ngrok auth token - get free at dashboard.ngrok.com
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"
ngrok.set_auth_token(NGROK_TOKEN)

# Start FastAPI in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

# Create public tunnel
public_url = ngrok.connect(8000)
API_URL = str(public_url)

print("FastAPI is running!")
print("Public URL: " + API_URL)
print("SAVE THIS URL - you need it for Day 4 Streamlit frontend")
print()
print("API Docs available at: " + API_URL + "/docs")

# Save URL to Drive for Day 4
with open(BASE + "/outputs/api_url.txt", "w") as f:
    f.write(API_URL)
print("URL saved to Drive!")


## STEP 11: Test the API

Now we test our API by sending real HTTP requests to it.
This is exactly how the Streamlit frontend will use it on Day 4.


In [ ]:
import requests as req

def test_api(question):
    response = req.post(
        API_URL + "/chat",
        json={"question": question, "use_graph": True},
        timeout=60
    )
    if response.status_code == 200:
        data = response.json()
        print("Question: " + question)
        print("Answer: " + data["answer"][:300])
        print("Sources:")
        for s in data["sources"]:
            print("  " + s["label"])
        print("Turn: " + str(data["turn"]))
    else:
        print("Error: " + str(response.status_code))
        print(response.text)

# Test health
health = req.get(API_URL + "/health")
print("Health: " + str(health.json()))

# Test stats
stats = req.get(API_URL + "/stats")
print("Stats: " + str(stats.json()))
print()

# Test chat
print("=" * 50)
test_api("What is Tesla revenue in 2023?")
print()
test_api("What are Tesla main vehicle products?")


## STEP 12: Save Everything for Day 4


In [ ]:
summary_d3 = {
    "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "components_built": [
        "neo4j_knowledge_graph",
        "entity_extraction",
        "graph_enhanced_search",
        "full_rag_with_graph",
        "conversational_rag_upgraded",
        "fastapi_backend_4_endpoints",
        "ngrok_public_url"
    ],
    "api_url": API_URL,
    "graph_stats": kg.get_graph_stats(),
    "status": "DAY 3 COMPLETE"
}

with open(BASE + "/outputs/day3_summary.json", "w") as f:
    json.dump(summary_d3, f, indent=2)

print("=" * 50)
print("DAY 3 COMPLETE!")
print("=" * 50)
print(json.dumps(summary_d3, indent=2))
print()
print("IMPORTANT: Save this API URL for Day 4:")
print(API_URL)
print()
print("File > Download > Download .ipynb")
print("Commit: [Day-15] Enterprise Knowledge Navigator - Day3 GraphRAG FastAPI")


## DAY 3 COMPLETE!

### What you built:
- Neo4j knowledge graph with entity extraction from documents
- Graph-enhanced search using entity relationships
- Full RAG pipeline upgraded with Graph RAG
- ConversationalRAG class with 10-turn memory
- FastAPI backend with 4 endpoints
- Public URL via ngrok for frontend connection

### Key concepts learned:
1. Graph RAG: entities and relationships enable multi-hop reasoning
2. Entity extraction: LLM identifies structured data from unstructured text
3. REST API: how backends expose functionality to frontends
4. Two-tier architecture: FastAPI backend + Streamlit frontend

### Before Day 4:
1. Save the ngrok API URL printed above
2. Download this notebook
3. Push to GitHub

### Day 4 preview:
- Streamlit chat frontend with source highlighting
- RAGAS evaluation dashboard
- Final GitHub cleanup and README
- Project complete!
